# Tutorial: Decision Tree on Paper - Should You Play Outside?

Audience:
- Students learning the basic idea of decision trees.

Prerequisites:
- Basic Python.
- The idea that data can be split into smaller groups.

Learning goals:
- Build a small decision tree from labeled examples.
- Draw the tree in a clean paper-style format.
- Use the tree to predict whether you should play outside.


## Outline

1. Create a tiny weather dataset.
2. Measure which feature gives the best split.
3. Build the decision tree.
4. Draw the tree and test a few predictions.
5. Try one short exercise.


In [ ]:
from __future__ import annotations

from collections import Counter
from math import log2
from pprint import pprint

training_data = [
    {"Weather": "Sunny", "Windy": "No", "Temperature": "Warm", "Play": "Yes"},
    {"Weather": "Sunny", "Windy": "Yes", "Temperature": "Warm", "Play": "No"},
    {"Weather": "Sunny", "Windy": "No", "Temperature": "Cold", "Play": "Yes"},
    {"Weather": "Rainy", "Windy": "No", "Temperature": "Warm", "Play": "No"},
    {"Weather": "Rainy", "Windy": "Yes", "Temperature": "Cold", "Play": "No"},
    {"Weather": "Cloudy", "Windy": "No", "Temperature": "Warm", "Play": "Yes"},
    {"Weather": "Cloudy", "Windy": "Yes", "Temperature": "Cold", "Play": "Yes"},
    {"Weather": "Sunny", "Windy": "Yes", "Temperature": "Cold", "Play": "No"}
]

features = ["Weather", "Windy", "Temperature"]
target = "Play"

pprint(training_data)


## Step 1 - Decide how to choose a split

A decision tree asks a question at each node. To choose the best question, we compute **information gain**. The feature with the highest gain becomes the next split.


In [ ]:
def entropy(rows: list[dict[str, str]], label: str) -> float:
    counts = Counter(row[label] for row in rows)
    total = len(rows)
    return -sum((count / total) * log2(count / total) for count in counts.values())


def split_rows(rows: list[dict[str, str]], feature: str) -> dict[str, list[dict[str, str]]]:
    groups: dict[str, list[dict[str, str]]] = {}
    for row in rows:
        groups.setdefault(row[feature], []).append(row)
    return groups


def information_gain(rows: list[dict[str, str]], feature: str, label: str) -> float:
    base_entropy = entropy(rows, label)
    groups = split_rows(rows, feature)
    weighted_entropy = sum((len(group) / len(rows)) * entropy(group, label) for group in groups.values())
    return base_entropy - weighted_entropy


for feature in features:
    print(f"{feature}: information gain = {information_gain(training_data, feature, target):.3f}")


## Step 2 - Build the decision tree

This is a small ID3-style tree builder. If all examples in a group have the same answer, it makes a leaf node. Otherwise, it picks the feature with the highest information gain and splits again.


In [ ]:
def majority_label(rows: list[dict[str, str]], label: str) -> str:
    return Counter(row[label] for row in rows).most_common(1)[0][0]


def build_tree(rows: list[dict[str, str]], remaining_features: list[str], label: str):
    labels = [row[label] for row in rows]
    if len(set(labels)) == 1:
        return labels[0]

    if not remaining_features:
        return majority_label(rows, label)

    best_feature = max(remaining_features, key=lambda feature: information_gain(rows, feature, label))
    tree = {best_feature: {}}
    next_features = [feature for feature in remaining_features if feature != best_feature]

    for feature_value, subset in split_rows(rows, best_feature).items():
        tree[best_feature][feature_value] = build_tree(subset, next_features, label)

    return tree


def draw_tree(tree, indent: str = "") -> str:
    if isinstance(tree, str):
        return indent + f"Answer: {tree}"

    lines: list[str] = []
    feature = next(iter(tree))
    lines.append(indent + f"[{feature}]?")

    for feature_value, branch in tree[feature].items():
        lines.append(indent + f"  -> {feature_value}")
        lines.append(draw_tree(branch, indent + "     "))

    return "\n".join(lines)


decision_tree = build_tree(training_data, features, target)
print(draw_tree(decision_tree))


## Step 3 - Use the tree for prediction

Now we can pass in a new weather situation and let the tree decide whether the answer should be `Yes` or `No`.


In [ ]:
def predict(tree, sample: dict[str, str]) -> str:
    if isinstance(tree, str):
        return tree

    feature = next(iter(tree))
    value = sample[feature]
    branch = tree[feature].get(value)

    if branch is None:
        return "Unknown"

    return predict(branch, sample)


test_samples = [
    {"Weather": "Sunny", "Windy": "No", "Temperature": "Warm"},
    {"Weather": "Sunny", "Windy": "Yes", "Temperature": "Cold"},
    {"Weather": "Cloudy", "Windy": "Yes", "Temperature": "Warm"},
    {"Weather": "Rainy", "Windy": "No", "Temperature": "Cold"}
]

for sample in test_samples:
    print(sample, "->", predict(decision_tree, sample))


## Paper-style final answer

A neat version of the learned tree is:

```text
[Weather]?
  -> Cloudy   : Yes
  -> Rainy    : No
  -> Sunny    : check [Windy]?
       -> No  : Yes
       -> Yes : No
```

So the prediction rule is simple:
- If the weather is cloudy, play outside.
- If the weather is rainy, do not play outside.
- If the weather is sunny, play only when it is not windy.


## Exercises

- Predict the answer for `Weather = Sunny`, `Windy = No`, `Temperature = Cold` before running the code.
- Change one training example and see whether the root feature changes.
- Common mistake: using too few examples can make a tree look certain when it is really fragile.
- Extension: add a new feature such as `Humidity` and rebuild the tree.


In [ ]:
# Exercise answer scaffold
my_sample = {"Weather": "Sunny", "Windy": "No", "Temperature": "Cold"}
predict(decision_tree, my_sample)
